In [3]:
import requests
import pandas as pd
import re
import os
import time
import random
import io
from datetime import datetime, timedelta

In [11]:
# Test
res = requests.post(
    "https://traffic.dot.ga.gov/ATSPM/DefaultCharts/GetTMCMetric",
    json={
        "SignalID": "6102",
        "StartDate": "11/16/2025 12:00 AM",
        "EndDate": "11/16/2025 11:59 PM",
        "YAxisMax": "",
        "Y2AxisMax": "",
        "MetricTypeID": 5,
        "SelectedBinSize": "60",
        "ShowLaneVolumes": True,
        "ShowTotalVolumes": True,
        "ShowDataTable": True,
    }
)

print(res)
#print(res.text)

<Response [200]>


In [9]:
# -----------------------------
# Config
# -----------------------------
URL = "https://traffic.dot.ga.gov/ATSPM/DefaultCharts/GetTMCMetric"

SIGNAL_IDS = ['1234','7730', '6455']

START_DATE = "5/01/2025 12:00 AM"
END_DATE   = "11/01/2025 11:59 PM"

OUTPUT_DIR = "signal_excels"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Regex to extract TMCTable
TMCTABLE_RE = re.compile(
    r'<div class="TMCTable">(.*?)</div>',
    re.DOTALL | re.IGNORECASE
)

# -----------------------------
# Helper
# -----------------------------
def split_date_range(start_str, end_str):
    start_dt = datetime.strptime(start_str, "%m/%d/%Y %I:%M %p")
    end_dt = datetime.strptime(end_str, "%m/%d/%Y %I:%M %p")
    ranges = []
    current_start = start_dt
    while current_start < end_dt:
        current_end = min(current_start + timedelta(days=7) - timedelta(seconds=1), end_dt)
        ranges.append((current_start, current_end))
        current_start = current_end + timedelta(seconds=1)
    return ranges

# -----------------------------
# API Call
# -----------------------------
for sid in SIGNAL_IDS:
    
    # ---------------------------------------------
    # Skip if output file already exists
    # ---------------------------------------------
    output_path = os.path.join(OUTPUT_DIR, f"tmc_{sid}.xlsx")
    if os.path.exists(output_path):
        print(f"Skipping Signal {sid}: file already exists ({output_path})")
        continue
    # ---------------------------------------------
    
    print(f"\nFetching TMC for signal {sid}...")
    all_dfs = []

    for start_dt, end_dt in split_date_range(START_DATE, END_DATE):
        payload = {
            "SignalID": sid,
            "StartDate": start_dt.strftime("%m/%d/%Y %I:%M %p"),
            "EndDate": end_dt.strftime("%m/%d/%Y %I:%M %p"),
            "YAxisMax": "",
            "Y2AxisMax": "",
            "MetricTypeID": 5,
            "SelectedBinSize": "60",
            "ShowLaneVolumes": True,
            "ShowTotalVolumes": True,
            "ShowDataTable": True,
        }

        try:
            res = requests.post(URL, json=payload)
            res.raise_for_status()
            html = res.text

            # Extract TMCTable block
            match = TMCTABLE_RE.search(html)
            if not match:
                print(f"No TMCTable found for {sid} ({payload['StartDate']} - {payload['EndDate']}), skipping.")
                continue

            tmc_html = match.group(1)
            # Extract table element
            table_match = re.search(r"<table.*?>.*?</table>", tmc_html, re.DOTALL | re.IGNORECASE)
            if not table_match:
                print(f"No table found for {sid}, skipping.")
                continue

            table_html = table_match.group(0)
            df = pd.read_html(io.StringIO(table_html))[0]
            df = df.iloc[:-1]

            # Flatten multi-row headers if needed
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = [
                    "_".join([str(c) for c in col if "Unnamed" not in str(c)])
                    for col in df.columns
                ]
            df.columns = [col.strip().replace(" ", "_") for col in df.columns]
            df["SignalID"] = sid
            df["Datetime"] = pd.date_range(start=start_dt, periods=len(df), freq='h')
            
            all_dfs.append(df)
            print(f"   ✓ Collected {len(df)} rows ({payload['StartDate']} - {payload['EndDate']})")

            # Random delay 1-5 seconds
            time.sleep(random.uniform(1, 5))

        except Exception as e:
            print(f"Error for {sid} ({payload['StartDate']} - {payload['EndDate']}): {e}")
            # Save partial results so far
            if all_dfs:
                partial_path = os.path.join(OUTPUT_DIR, f"tmc_{sid}_partial.xlsx")
                pd.concat(all_dfs, ignore_index=True).to_excel(partial_path, index=False)
                print(f"Saved partial data to {partial_path}")
            continue

    # Save final Excel for this signal
    if all_dfs:
        out_path = os.path.join(OUTPUT_DIR, f"tmc_{sid}.xlsx")
        pd.concat(all_dfs, ignore_index=True).to_excel(out_path, index=False)
        print(f"Saved final Excel: {out_path} ({sum(len(df) for df in all_dfs)} rows)")

print("\nDone.")



Fetching TMC for signal 1234...
   ✓ Collected 168 rows (05/01/2025 12:00 AM - 05/07/2025 11:59 PM)
   ✓ Collected 168 rows (05/08/2025 12:00 AM - 05/14/2025 11:59 PM)
   ✓ Collected 168 rows (05/15/2025 12:00 AM - 05/21/2025 11:59 PM)
   ✓ Collected 168 rows (05/22/2025 12:00 AM - 05/28/2025 11:59 PM)
   ✓ Collected 168 rows (05/29/2025 12:00 AM - 06/04/2025 11:59 PM)
   ✓ Collected 168 rows (06/05/2025 12:00 AM - 06/11/2025 11:59 PM)
   ✓ Collected 168 rows (06/12/2025 12:00 AM - 06/18/2025 11:59 PM)
   ✓ Collected 168 rows (06/19/2025 12:00 AM - 06/25/2025 11:59 PM)
   ✓ Collected 168 rows (06/26/2025 12:00 AM - 07/02/2025 11:59 PM)
   ✓ Collected 168 rows (07/03/2025 12:00 AM - 07/09/2025 11:59 PM)
   ✓ Collected 168 rows (07/10/2025 12:00 AM - 07/16/2025 11:59 PM)
   ✓ Collected 168 rows (07/17/2025 12:00 AM - 07/23/2025 11:59 PM)
   ✓ Collected 168 rows (07/24/2025 12:00 AM - 07/30/2025 11:59 PM)
   ✓ Collected 168 rows (07/31/2025 12:00 AM - 08/06/2025 11:59 PM)
   ✓ Collected 